# The Golden Dome Heist Query Challenge — SQL Walkthrough

Now that you've solved the case by hand, let's see how SQL would have done the same detective work. Each section below maps directly to a clue from the activity.

We'll use **SQLite** — a lightweight database engine built into Python — so there's nothing extra to install.

---
## Setup: Building Our Database

First, we create the same five tables you just worked with on paper and load the data.

In [1]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# We'll use this helper throughout to run a query and display it as a nice table
def run(query):
    """Execute a SQL query and return results as a DataFrame."""
    return pd.read_sql_query(query, conn)

In [2]:
# ── Create Tables ──

cursor.executescript("""
CREATE TABLE suspects (
    suspect_id     INTEGER PRIMARY KEY,
    first_name     TEXT,
    last_name      TEXT,
    department     TEXT,
    has_key_access TEXT
);

CREATE TABLE locations (
    location_id   INTEGER PRIMARY KEY,
    building_name TEXT,
    floor         TEXT
);

CREATE TABLE sightings (
    sighting_id  INTEGER PRIMARY KEY,
    suspect_id   INTEGER,
    location_id  INTEGER,
    time_seen    TEXT,
    FOREIGN KEY (suspect_id)  REFERENCES suspects(suspect_id),
    FOREIGN KEY (location_id) REFERENCES locations(location_id)
);

CREATE TABLE evidence (
    evidence_id TEXT PRIMARY KEY,
    description TEXT,
    location_id INTEGER,
    FOREIGN KEY (location_id) REFERENCES locations(location_id)
);

CREATE TABLE alibis (
    alibi_id          TEXT PRIMARY KEY,
    suspect_id        INTEGER,
    alibi_description TEXT,
    verified          TEXT,
    FOREIGN KEY (suspect_id) REFERENCES suspects(suspect_id)
);
""")

print("Tables created successfully!")

Tables created successfully!


In [3]:
# ── Insert Data ──

cursor.executemany("INSERT INTO suspects VALUES (?, ?, ?, ?, ?)", [
    (1, "Maria",  "Chen",     "Computer Science", "Yes"),
    (2, "James",  "O'Brien",  "History",          "No"),
    (3, "Sofia",  "Ramirez",  "Engineering",      "Yes"),
    (4, "Derek",  "Kim",      "Business",         "Yes"),
    (5, "Ava",    "Thompson", "Biology",           "Yes"),
    (6, "Marcus", "Wells",    "Art",               "No"),
    (7, "Priya",  "Patel",    "Chemistry",         "Yes"),
    (8, "Tyler",  "Johnson",  "Athletics",         "Yes"),
])

cursor.executemany("INSERT INTO locations VALUES (?, ?, ?)", [
    (101, "Main Hall",       "1"),
    (102, "Library",         "2"),
    (103, "Science Lab",     "1"),
    (104, "Dining Hall",     "1"),
    (105, "Gymnasium",       "1"),
    (106, "Parking Garage",  "B1"),
])

cursor.executemany("INSERT INTO sightings VALUES (?, ?, ?, ?)", [
    (1,  1, 102, "7:30 PM"),
    (2,  2, 104, "8:15 PM"),
    (3,  3, 101, "8:45 PM"),
    (4,  4, 101, "8:05 PM"),
    (5,  5, 103, "8:30 PM"),
    (6,  6, 101, "8:20 PM"),
    (7,  7, 104, "9:00 PM"),
    (8,  8, 101, "8:40 PM"),
    (9,  3, 106, "9:10 PM"),
    (10, 6, 104, "7:45 PM"),
    (11, 1, 102, "8:50 PM"),
])

cursor.executemany("INSERT INTO evidence VALUES (?, ?, ?)", [
    ("E1", "Broken glass near display case",         101),
    ("E2", "Muddy footprints (size 11 men's shoe)",  101),
    ("E3", "Dropped university keycard",              106),
    ("E4", "Security camera footage (corrupted)",     101),
    ("E5", "Gray glove fibers on display case",       101),
])

cursor.executemany("INSERT INTO alibis VALUES (?, ?, ?, ?)", [
    ("A1", 1, "Studying in library – confirmed by librarian",       "Yes"),
    ("A2", 2, "At dinner with friends – confirmed by 3 witnesses",  "Yes"),
    ("A3", 3, "Claims was in office working late – no witnesses",    "No"),
    ("A4", 4, "At gym – membership card scanned at 7:00 PM",        "Yes"),
    ("A5", 5, "Running experiment in lab – lab partner confirms",    "Yes"),
    ("A6", 6, "Says was at home watching TV – no witnesses",         "No"),
    ("A7", 7, "At dining hall – receipt timestamped 8:55 PM",        "Yes"),
    ("A8", 8, "Getting something from car in garage – no witnesses", "No"),
])

conn.commit()
print("All data loaded!")

All data loaded!


---
## Quick Look: What's in each table?

Before we start investigating, let's peek at our data. `SELECT *` means "give me every column."

In [4]:
run("SELECT * FROM suspects")

,suspect_id,first_name,last_name,department,has_key_access
0,1,Maria,Chen,Computer Science,Yes
1,2,James,O'Brien,History,No
2,3,Sofia,Ramirez,Engineering,Yes
3,4,Derek,Kim,Business,Yes
4,5,Ava,Thompson,Biology,Yes
5,6,Marcus,Wells,Art,No
6,7,Priya,Patel,Chemistry,Yes
7,8,Tyler,Johnson,Athletics,Yes


In [5]:
run("SELECT * FROM locations")

,location_id,building_name,floor
0,101,Main Hall,1
1,102,Library,2
2,103,Science Lab,1
3,104,Dining Hall,1
4,105,Gymnasium,1
5,106,Parking Garage,B1


In [6]:
run("SELECT * FROM sightings")

,sighting_id,suspect_id,location_id,time_seen
0,1,1,102,7:30 PM
1,2,2,104,8:15 PM
2,3,3,101,8:45 PM
3,4,4,101,8:05 PM
4,5,5,103,8:30 PM
5,6,6,101,8:20 PM
6,7,7,104,9:00 PM
7,8,8,101,8:40 PM
8,9,3,106,9:10 PM
9,10,6,104,7:45 PM


In [7]:
run("SELECT * FROM evidence")

,evidence_id,description,location_id
0,E1,Broken glass near display case,101
1,E2,Muddy footprints (size 11 men's shoe),101
2,E3,Dropped university keycard,106
3,E4,Security camera footage (corrupted),101
4,E5,Gray glove fibers on display case,101


In [8]:
run("SELECT * FROM alibis")

,alibi_id,suspect_id,alibi_description,verified
0,A1,1,Studying in library – confirmed by librarian,Yes
1,A2,2,At dinner with friends – confirmed by 3 witnesses,Yes
2,A3,3,Claims was in office working late – no witnesses,No
3,A4,4,At gym – membership card scanned at 7:00 PM,Yes
4,A5,5,Running experiment in lab – lab partner confirms,Yes
5,A6,6,Says was at home watching TV – no witnesses,No
6,A7,7,At dining hall – receipt timestamped 8:55 PM,Yes
7,A8,8,Getting something from car in garage – no witn...,No


---
## Clue 1: Who Was at the Scene?

**What you did by hand:** You walked to the sightings table, found rows where the location was Main Hall (101), and filtered for times between 8:00 and 9:00 PM.

**SQL concepts:** `SELECT`, `FROM`, `WHERE`

**Result:** 4 suspects → **suspect_id 3, 4, 6, 8**

In [10]:
run("""
SELECT suspect_id, time_seen
FROM sightings
WHERE location_id = 101
  AND time_seen BETWEEN '8:00 PM' AND '9:00 PM'
""")

,suspect_id,time_seen
0,3,8:45 PM
1,4,8:05 PM
2,6,8:20 PM
3,8,8:40 PM


We found **four suspects** at Main Hall during the crime window. But these are just ID numbers — we need names. During the activity, you had to physically walk over to the suspects table and look them up. In SQL, we use a **JOIN**.

In [11]:
# Let's get their  names by joining with the suspects table
run("""
SELECT s.first_name, s.last_name, si.time_seen
FROM sightings si
JOIN suspects s ON si.suspect_id = s.suspect_id
WHERE si.location_id = 101
  AND si.time_seen BETWEEN '8:00 PM' AND '9:00 PM'
""")

,first_name,last_name,time_seen
0,Sofia,Ramirez,8:45 PM
1,Derek,Kim,8:05 PM
2,Marcus,Wells,8:20 PM
3,Tyler,Johnson,8:40 PM


---
## Clue 2: Who Had Key Access?

**What you did by hand:** You took those suspect_ids from sightings and looked them up in the suspects table to check key access.

**SQL concepts:** `JOIN ... ON` (connecting two tables using a shared key), additional `WHERE` conditions

**Result:** 4 → 3 suspects (Marcus Wells eliminated — no key access)

### What is a JOIN?

A `JOIN` connects rows from two tables based on a matching column — exactly like when you took a `suspect_id` from one sheet and found the same ID on another sheet.

In [11]:
# First, let's see EVERYONE at the scene with their key access status
run("""
SELECT s.first_name, s.last_name, s.department, s.has_key_access, si.time_seen
FROM sightings si
JOIN suspects s ON si.suspect_id = s.suspect_id
WHERE si.location_id = 101
  AND si.time_seen BETWEEN '8:00 PM' AND '9:00 PM'
""")

,first_name,last_name,department,has_key_access,time_seen
0,Sofia,Ramirez,Engineering,Yes,8:45 PM
1,Derek,Kim,Business,Yes,8:05 PM
2,Marcus,Wells,Art,No,8:20 PM
3,Tyler,Johnson,Athletics,Yes,8:40 PM


In [12]:
# Now filter to only those WITH key access
run("""
SELECT s.first_name, s.last_name, s.department, s.has_key_access, si.time_seen
FROM sightings si
JOIN suspects s ON si.suspect_id = s.suspect_id
WHERE si.location_id = 101
  AND si.time_seen BETWEEN '8:00 PM' AND '9:00 PM'
  AND s.has_key_access = 'Yes'
""")

,first_name,last_name,department,has_key_access,time_seen
0,Sofia,Ramirez,Engineering,Yes,8:45 PM
1,Derek,Kim,Business,Yes,8:05 PM
2,Tyler,Johnson,Athletics,Yes,8:40 PM


Marcus Wells is eliminated — no key access. Three suspects remain: **Sofia Ramirez**, **Derek Kim**, and **Tyler Johnson**.

---
## Clue 3: Checking Alibis

**What you did by hand:** You went to the alibis table and looked up each remaining suspect by their suspect_id to see if their alibi was verified.

**SQL concepts:** Another `JOIN`, `IN` (matching multiple values)

**Result:** 3 → 2 suspects (Derek Kim eliminated — verified gym alibi)

In [13]:
# Check alibis for all three remaining suspects
run("""
SELECT s.first_name, s.last_name, a.alibi_description, a.verified
FROM suspects s
JOIN alibis a ON s.suspect_id = a.suspect_id
WHERE s.suspect_id IN (3, 4, 8)
""")

,first_name,last_name,alibi_description,verified
0,Sofia,Ramirez,Claims was in office working late – no witnesses,No
1,Derek,Kim,At gym – membership card scanned at 7:00 PM,Yes
2,Tyler,Johnson,Getting something from car in garage – no witn...,No


Derek Kim's alibi checks out — his gym membership card was scanned at 7:00 PM, and it's verified. He's eliminated.

Two suspects remain with **unverified** alibis: **Sofia Ramirez** and **Tyler Johnson**.

In [14]:
# Filter to just the unverified alibis
run("""
SELECT s.first_name, s.last_name, a.alibi_description, a.verified
FROM suspects s
JOIN alibis a ON s.suspect_id = a.suspect_id
WHERE s.suspect_id IN (3, 4, 8)
  AND a.verified = 'No'
""")

,first_name,last_name,alibi_description,verified
0,Sofia,Ramirez,Claims was in office working late – no witnesses,No
1,Tyler,Johnson,Getting something from car in garage – no witn...,No


---
## Clue 4: Following the Evidence

**What you did by hand:** You checked the evidence table, found a dropped keycard at the Parking Garage, then went back to sightings to see which remaining suspect was seen there.

**SQL concepts:** `ORDER BY` (sorting results), joining multiple tables for readable output

**Result:** 2 → 1 suspect (Tyler Johnson has no Parking Garage sighting)

In [15]:
# What evidence was found at the Parking Garage?
run("""
SELECT e.evidence_id, e.description, l.building_name
FROM evidence e
JOIN locations l ON e.location_id = l.location_id
WHERE l.building_name = 'Parking Garage'
""")

,evidence_id,description,building_name
0,E3,Dropped university keycard,Parking Garage


In [16]:
# Reconstruct the timeline for BOTH remaining suspects
run("""
SELECT s.first_name, s.last_name, si.time_seen, l.building_name
FROM sightings si
JOIN locations l ON si.location_id = l.location_id
JOIN suspects s ON si.suspect_id = s.suspect_id
WHERE si.suspect_id IN (3, 8)
ORDER BY s.last_name, si.time_seen
""")

,first_name,last_name,time_seen,building_name
0,Tyler,Johnson,8:40 PM,Main Hall
1,Sofia,Ramirez,8:45 PM,Main Hall
2,Sofia,Ramirez,9:10 PM,Parking Garage


**The timeline tells the story:**

- **Tyler Johnson** was only seen at Main Hall at 8:40 PM — no connection to the Parking Garage.
- **Sofia Ramirez** was at Main Hall at 8:45 PM (crime scene, during the window), then at the Parking Garage at 9:10 PM (where the dropped keycard was found).

She was at the crime scene during the crime window, had key access, has no verified alibi, and fled to the parking garage where she dropped her keycard.

### **Sofia Ramirez stole the Golden Dome Trophy.**

---
## Putting It All Together: One Query to Crack the Case

What if we combined everything — scene presence, key access, unverified alibi, and flight to the parking garage — into a single query?

In [17]:
run("""
SELECT
    s.first_name,
    s.last_name,
    s.department,
    a.alibi_description,
    a.verified AS alibi_verified
FROM suspects s
-- They were at Main Hall during the crime window
JOIN sightings scene ON s.suspect_id = scene.suspect_id
    AND scene.location_id = 101
    AND scene.time_seen BETWEEN '8:00 PM' AND '9:00 PM'
-- They were also seen at the Parking Garage (where the keycard was found)
JOIN sightings flight ON s.suspect_id = flight.suspect_id
    AND flight.location_id = 106
-- Their alibi
JOIN alibis a ON s.suspect_id = a.suspect_id
-- They had key access and the alibi is unverified
WHERE s.has_key_access = 'Yes'
  AND a.verified = 'No'
""")

,first_name,last_name,department,alibi_description,alibi_verified
0,Sofia,Ramirez,Engineering,Claims was in office working late – no witnesses,No


One query. Four tables joined together. The database did in milliseconds what took your group 15 minutes of walking around the room.

That's the power of SQL.

---
## SQL Concepts Cheat Sheet

Here's a summary of every SQL keyword you just used:

| Keyword | What It Does | Activity Equivalent |
|---------|-------------|---------------------|
| `SELECT` | Choose which columns to display | "Which fields on the sheet did I look at?" |
| `FROM` | Specify which table to query | "Which station did I walk to?" |
| `WHERE` | Filter rows by a condition | "Which rows matched what I was looking for?" |
| `JOIN ... ON` | Connect two tables using a shared key | "I took an ID from one sheet and found it on another" |
| `AND` | Combine multiple conditions | "It has to match ALL of these criteria" |
| `IN (...)` | Match any value in a list | "Check these specific suspect IDs" |
| `BETWEEN` | Filter for a range of values | "Only times from 8:00 to 9:00" |
| `ORDER BY` | Sort results | "Let me arrange these by time" |
| `SELECT *` | Get all columns | "Show me the whole row" |

---
## Practice: Try It Yourself

Use the `run()` function to write your own queries. Here are some challenges:

1. **Find all suspects in the Biology or Chemistry departments.**
2. **List every sighting at the Dining Hall, showing suspect names and times (you'll need a JOIN).**
3. **Which suspects have verified alibis? Show their names and alibi descriptions.**
4. **Find all evidence found at Main Hall. Show the evidence description and building name.**
5. **CHALLENGE: Who was the last person seen at any location? Show their name, the building, and the time.**

In [18]:
# 1. Suspects in Biology or Chemistry



In [19]:
# 2. Every sighting at the Dining Hall with suspect names



In [20]:
# 3. Suspects with verified alibis



In [21]:
# 4. Evidence found at Main Hall



In [22]:
# 5. CHALLENGE: Last person seen at any location



In [23]:
# Clean up when done
conn.close()